# Full Segmentation Pipeline with Metadata Output

This notebook finishes Step 1. It takes the page-boundary and document-type prompts from the page-level detection notebook, runs them on the bundled pharmaceutical PDF with the local Mistral model chosen there, and writes the page metadata the LlamaIndex tagging notebook was attaching by hand.

Gemini is not used. Every page is classified by `mistral-7b-instruct-v0.2` (the GGUF already saved in `content/mistral.gguf`).

The result for each page is:

```json
{ "page": 0, "is_new_doc": "Yes", "doc_type": "Cover Letter", "page_in_doc": 0 }
```


**Step 1: Load the bundled PDF, one page at a time**

The input is `pharma-blob-sample.pdf`: certificates of quality, a packaging specification, a BSE/TSE declaration, and the other vendor documents stitched into one file. PyMuPDF reads each page's text. Page indexes stay zero-based so they match the metadata schema.


In [1]:
# ============================================
# STEP 1: Extract one text block per page
# ============================================
#
# WHAT WE'RE DOING:
# Opening the bundled vendor PDF with PyMuPDF and keeping
# each page's text next to its zero-based index.
#
# WHY THIS MATTERS:
# Boundary detection compares neighboring pages. If the
# pages are merged into one string, a new certificate
# looks like a continuation of the previous one.
#
# WHAT YOU'LL SEE:
# The page count and the first line of every page.
# ============================================

from pathlib import Path
import pymupdf

PDF_PATH = Path("pharma-blob-sample.pdf")
pdf = pymupdf.open(PDF_PATH)

doc_pages = []
for index, page in enumerate(pdf):
    text = page.get_text("text").strip()
    doc_pages.append({"page": index, "text": text})
pdf.close()

print(f"Loaded {len(doc_pages)} pages from {PDF_PATH.name}")
for page in doc_pages:
    first_line = next((line.strip() for line in page["text"].splitlines() if line.strip()), "(blank)")
    print(f"  page {page['page']:>2}  {len(page['text']):4d} chars  {first_line[:72]}")


Loaded 10 pages from pharma-blob-sample.pdf
  page  0  1238 chars  Cytiva
  page  1   958 chars  Certificate of Quality
  page  2   960 chars  Certificate of Quality
  page  3   723 chars  Packaging Component Specification
  page  4   746 chars  Packaging Component Specification (continued)
  page  5  1089 chars  Declaration Regarding Transmissible Spongiform
  page  6  1049 chars  Material Description Sheet
  page  7   946 chars  Supplier Qualification Record
  page  8   986 chars  Supplier Qualification Record (continued)
  page  9   992 chars  Cytiva


**Step 2: Load the open-source model**

The page-level notebook compared Gemini with local Mistral and kept Mistral Instruct v0.2 for this pipeline. The weights are the Q4_K_M GGUF already downloaded to `content/mistral.gguf`. All layers go to the GPU. The earlier notebook offloaded a single layer only to prove the file loaded.


In [2]:
# ============================================
# STEP 2: Load Mistral Instruct locally
# ============================================
#
# WHAT WE'RE DOING:
# Loading the same GGUF selected in the page-level notebook.
# No Gemini client is created in this file.
#
# WHY THIS MATTERS:
# Document labels for a vendor file should be reproducible
# without sending the pages to a hosted model.
#
# WHAT YOU'LL SEE:
# The resolved model path and the context size. Weight
# loading itself stays quiet.
# ============================================

from llama_cpp import Llama, LlamaGrammar

MODEL_PATH = Path("content/mistral.gguf")
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Missing content/mistral.gguf. That file is the "
        "TheBloke/Mistral-7B-Instruct-v0.2-GGUF Q4_K_M download "
        "used by the page-level notebook."
    )

llm = Llama(
    model_path=str(MODEL_PATH),
    n_gpu_layers=-1,
    n_ctx=4096,
    n_batch=512,
    verbose=False,
    seed=7,
)
print(f"Loaded {MODEL_PATH} ({MODEL_PATH.stat().st_size / (1024 ** 3):.2f} GB), n_ctx=4096")


Loaded content/mistral.gguf (4.07 GB), n_ctx=4096


**Test a Prompt Live**

This is the structured prompt for the two tasks: decide whether a page starts a document, and name its type. The snippet was written for Gemini. Here `mistral_chat` sends it to the local Mistral model, and `json.loads` reads the JSON object off the reply.


In [3]:
# ============================================
# Live prompt: boundary detection + classification
# ============================================
#
# WHAT WE'RE DOING:
# Sending the structured boundary prompt to local Mistral.
# The call stands in for gemini.chat. The reply is parsed
# with json.loads.
#
# WHY THIS MATTERS:
# This is the same two-part task the full file uses:
# does the page start a document, and what type is it.
#
# WHAT YOU'LL SEE:
# One JSON object for the sample certificate sentence.
# ============================================

import json

page_text = "This certificate confirms that the referenced product batch meets all quality specifications..."

# Step 1: Build the prompt string
prompt = f"""
You are a document boundary detection agent.

Your job is to:
1. Decide if this page starts a new document or continues the previous one.
2. If it starts a new document, classify its type.

Document type options: ["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration", "Material Description", "Supplier Qualification", "Chain of Custody", "Other"]

Return only a JSON response in the format:
{{
  "is_new_doc": "<Yes/No>",
  "doc_type": "<one of the above>"
}}

Page Content:
{page_text}

If unsure about document type, return "Other".
"""

# The grammar keeps the reply inside that JSON shape so json.loads
# receives a finished object. A cutoff is not repaired into a label.
_LIVE_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other",
]
_yes_no = " | ".join('"' + '\\"' + choice + '\\"' + '"' for choice in ["Yes", "No"])
_types = " | ".join('"' + '\\"' + choice + '\\"' + '"' for choice in _LIVE_TYPES)
LIVE_GRAMMAR = LlamaGrammar.from_string(
    'root ::= "{" ws "\\"is_new_doc\\"" ws ":" ws yesno ws "," ws "\\"doc_type\\"" ws ":" ws doctype ws "}"\n'
    "ws ::= [ \\t\\n]*\n"
    "yesno ::= " + _yes_no + "\n"
    "doctype ::= " + _types + "\n"
)


def mistral_chat(prompt):
    """Local replacement for gemini.chat. The reply text is on .text."""
    completion = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=64,
        seed=7,
        grammar=LIVE_GRAMMAR,
    )
    text = completion["choices"][0]["message"]["content"]
    return type("MistralResponse", (), {"text": text})()


# Step 2: Call the model
response = mistral_chat(prompt)

# Step 3: Parse the response as JSON
output = json.loads(response.text)
output


{'is_new_doc': 'Yes', 'doc_type': 'Certificate of Quality'}

**Step 3: Ask one question per page, and check the answer**

The live prompt above asks both questions and returns one JSON object. On the full file, Mistral answers them one at a time, using the same type list and the same `is_new_doc` / `doc_type` fields. A continuation page keeps the open document's type.

```json
{ "is_new_doc": "Yes" }
```

```json
{ "doc_type": "Certificate of Quality" }
```

The boundary question uses ordered rules. A heading that says "continued" or "Page 2 of 2" stays in the open document. A different first heading starts a new one, even when a part number from the previous page shows up again. Two certificates with the same title and different lot numbers are two documents.

The type question runs only after a Yes. Its labels are the list from the tagging notebook, including the headings this packet actually prints ("Packaging Component Specification", "Material Description Sheet", and the others).

A reply that is cut off is not repaired into a label. The page is asked once more. If that also fails, a later page stays with the document already open. The first page can only start a document.


In [4]:
# ============================================
# STEP 3: Classify the page boundary and type
# ============================================
#
# WHAT WE'RE DOING:
# Asking Mistral two short questions, the same split as
# the page-level notebook. The first question only decides
# is_new_doc. The type question runs when the answer is Yes.
#
# WHY THIS MATTERS:
# One combined question kept the previous type alive across
# a page whose title had already changed. A new title has to
# win before the label is chosen.
#
# WHAT YOU'LL SEE:
# Nothing yet. classify_page is called in the next step.
# ============================================

import json

DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other",
]


def _enum_grammar(key, choices):
    """GBNF so the reply can only be one JSON field from a fixed list."""
    options = " | ".join('"' + '\\"' + choice + '\\"' + '"' for choice in choices)
    source = (
        'root ::= "{" ws "\\"' + key + '\\"" ws ":" ws value ws "}"\n'
        "ws ::= [ \\t\\n]*\n"
        "value ::= " + options + "\n"
    )
    return LlamaGrammar.from_string(source)


BOUNDARY_GRAMMAR = _enum_grammar("is_new_doc", ["Yes", "No"])
TYPE_GRAMMAR = _enum_grammar("doc_type", DOC_TYPES)


def _lines(text):
    return [line.strip() for line in text.splitlines() if line.strip()]


def _opening(text, n=8):
    return "\n".join(_lines(text)[:n])


def _complete(prompt, grammar):
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=32,
        seed=7,
        grammar=grammar,
    )
    choice = response["choices"][0]
    return choice["message"]["content"], choice.get("finish_reason")


def _ask_json(prompt, grammar, key, allowed):
    """Ask once, and once more if the reply is cut off or the wrong shape.

    A truncated reply is rejected. Closing its braces would invent a label.
    """
    last_error = "no reply"
    for _attempt in range(2):
        raw, finish_reason = _complete(prompt, grammar)
        if finish_reason == "length":
            last_error = "truncated reply"
            prompt += "\nThe last reply was cut off. Return only the JSON object."
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError as exc:
            last_error = str(exc)
            prompt += "\nThe last reply was not JSON. Return only the JSON object."
            continue
        value = data.get(key)
        if value not in allowed:
            last_error = f"{key}={value!r} is not an allowed value"
            prompt += "\nThe last reply used a value outside the list. Return only the JSON object."
            continue
        return value, False, None
    return None, True, last_error


def _boundary_prompt(page_index, curr_text, prev_text):
    curr_heading = _lines(curr_text)[0]
    if prev_text is None:
        prev_heading = "(none)"
        previous = "(none — this is the first page of the file)"
    else:
        prev_heading = _lines(prev_text)[0]
        previous = _opening(prev_text)
    return f"""Apply these rules in order. Stop at the first rule that matches.
1. If there is no previous page, is_new_doc is Yes.
2. If the current first heading contains "continued", is_new_doc is No. This wins even when the heading is longer than the previous heading.
3. If the current first heading is different from the previous first heading, is_new_doc is Yes.
4. If the headings match but the lot number, document number, or part number changed, is_new_doc is Yes.
5. Otherwise is_new_doc is No.

Previous first heading: {prev_heading}
Current first heading: {curr_heading}

Previous opening:
{previous}

Current opening:
{_opening(curr_text)}

Reply with JSON only.
"""


def _type_prompt(curr_text):
    return f"""Classify this pharmaceutical page into exactly one document type.
- Cover Letter: a letter, often "To Whom It May Concern", about storage or product information
- Certificate of Quality: lot number, manufacture date, expiration, and release criteria. The title is often "Certificate of Quality"
- Packaging Specification: packaging components and materials. The title may be "Packaging Component Specification"
- BSE/TSE Declaration: animal-origin materials or a transmissible spongiform encephalopathy compliance statement
- Material Description: materials of construction. The title may be "Material Description Sheet"
- Supplier Qualification: audits and ISO certificates. The title may be "Supplier Qualification Record"
- Chain of Custody: manufactured assemblies and the path from manufacturing to shipment. The title may be "Global Chain of Custody"
- Other: nothing above fits

Page:
{curr_text}

Reply with JSON only: {{"doc_type": "<one type from the list>"}}
"""


def classify_page(page_index, curr_text, prev_text, current_doc_type):
    """Return (is_new_doc, doc_type, used_fallback).

    A blank page, or a reply that still fails after one retry, does not
    open a new document of an invented type. It stays with the document
    already open. The first page is the exception: it has to start one.
    """
    if not curr_text.strip():
        if page_index == 0 or not current_doc_type:
            return "Yes", "Other", True
        return "No", current_doc_type, True

    is_new, failed, error = _ask_json(
        _boundary_prompt(page_index, curr_text, prev_text),
        BOUNDARY_GRAMMAR,
        "is_new_doc",
        ("Yes", "No"),
    )
    if failed:
        print(f"  boundary fallback on page {page_index}: {error}")
        if page_index == 0 or not current_doc_type:
            return "Yes", "Other", True
        return "No", current_doc_type, True

    if page_index == 0:
        is_new = "Yes"
    if is_new == "No":
        if not current_doc_type:
            return "Yes", "Other", True
        return "No", current_doc_type, False

    doc_type, failed, error = _ask_json(
        _type_prompt(curr_text),
        TYPE_GRAMMAR,
        "doc_type",
        tuple(DOC_TYPES),
    )
    if failed:
        print(f"  type fallback on page {page_index}: {error}")
        return "Yes", "Other", True
    return "Yes", doc_type, False


**Step 4: Walk the file and number the pages inside each document**

`page_in_doc` starts at 0 when `is_new_doc` is Yes. A following page that stays in that document increments it. The next Yes resets the counter.


In [5]:
# ============================================
# STEP 4: Build page-level metadata
# ============================================
#
# WHAT WE'RE DOING:
# Classifying every page, then tracking where each
# document starts so page_in_doc can reset.
#
# WHY THIS MATTERS:
# A reviewer needs to know both the type and which
# page of that document they are looking at.
#
# WHAT YOU'LL SEE:
# One metadata row per page, printed as it is decided.
# ============================================

results = []
current_doc_type = None
page_in_doc = 0

for index, page in enumerate(doc_pages):
    prev_text = None if index == 0 else doc_pages[index - 1]["text"]
    is_new, doc_type, used_fallback = classify_page(
        index, page["text"], prev_text, current_doc_type
    )
    if is_new == "Yes":
        page_in_doc = 0
        current_doc_type = doc_type
    else:
        page_in_doc += 1
        doc_type = current_doc_type

    row = {
        "page": index,
        "is_new_doc": is_new,
        "doc_type": doc_type,
        "page_in_doc": page_in_doc,
    }
    results.append(row)
    note = " (fallback)" if used_fallback else ""
    print(f"{row}{note}")


{'page': 0, 'is_new_doc': 'Yes', 'doc_type': 'Cover Letter', 'page_in_doc': 0}


{'page': 1, 'is_new_doc': 'Yes', 'doc_type': 'Certificate of Quality', 'page_in_doc': 0}


{'page': 2, 'is_new_doc': 'Yes', 'doc_type': 'Certificate of Quality', 'page_in_doc': 0}


{'page': 3, 'is_new_doc': 'Yes', 'doc_type': 'Packaging Specification', 'page_in_doc': 0}


{'page': 4, 'is_new_doc': 'No', 'doc_type': 'Packaging Specification', 'page_in_doc': 1}


{'page': 5, 'is_new_doc': 'Yes', 'doc_type': 'BSE/TSE Declaration', 'page_in_doc': 0}


{'page': 6, 'is_new_doc': 'Yes', 'doc_type': 'Material Description', 'page_in_doc': 0}


{'page': 7, 'is_new_doc': 'Yes', 'doc_type': 'Supplier Qualification', 'page_in_doc': 0}


{'page': 8, 'is_new_doc': 'No', 'doc_type': 'Supplier Qualification', 'page_in_doc': 1}


{'page': 9, 'is_new_doc': 'Yes', 'doc_type': 'Chain of Custody', 'page_in_doc': 0}


**Step 5: Write the metadata as a table and as JSON**

Both shapes carry the same four fields. The JSON list is what a later automation step can load. The DataFrame is the same records for review.


In [6]:
# ============================================
# STEP 5: Publish the metadata
# ============================================
#
# WHAT WE'RE DOING:
# Showing the segmentation as a JSON list and as a
# pandas DataFrame with the required columns.
#
# WHY THIS MATTERS:
# The list is the contract for the next pipeline stage.
# The table is the same contract, easier to scan.
#
# WHAT YOU'LL SEE:
# The JSON list, then the DataFrame.
# ============================================

import pandas as pd

print(json.dumps(results, indent=2))
metadata_df = pd.DataFrame(results)[["page", "is_new_doc", "doc_type", "page_in_doc"]]
metadata_df


[
  {
    "page": 0,
    "is_new_doc": "Yes",
    "doc_type": "Cover Letter",
    "page_in_doc": 0
  },
  {
    "page": 1,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 2,
    "is_new_doc": "Yes",
    "doc_type": "Certificate of Quality",
    "page_in_doc": 0
  },
  {
    "page": 3,
    "is_new_doc": "Yes",
    "doc_type": "Packaging Specification",
    "page_in_doc": 0
  },
  {
    "page": 4,
    "is_new_doc": "No",
    "doc_type": "Packaging Specification",
    "page_in_doc": 1
  },
  {
    "page": 5,
    "is_new_doc": "Yes",
    "doc_type": "BSE/TSE Declaration",
    "page_in_doc": 0
  },
  {
    "page": 6,
    "is_new_doc": "Yes",
    "doc_type": "Material Description",
    "page_in_doc": 0
  },
  {
    "page": 7,
    "is_new_doc": "Yes",
    "doc_type": "Supplier Qualification",
    "page_in_doc": 0
  },
  {
    "page": 8,
    "is_new_doc": "No",
    "doc_type": "Supplier Qualification",
    "page_in_doc": 1
  },
  {
 

,page,is_new_doc,doc_type,page_in_doc
0,0,Yes,Cover Letter,0
1,1,Yes,Certificate of Quality,0
2,2,Yes,Certificate of Quality,0
3,3,Yes,Packaging Specification,0
4,4,No,Packaging Specification,1
5,5,Yes,BSE/TSE Declaration,0
6,6,Yes,Material Description,0
7,7,Yes,Supplier Qualification,0
8,8,No,Supplier Qualification,1
9,9,Yes,Chain of Custody,0


**Step 6: Attach the labels in the LlamaIndex metadata shape**

The tagging notebook stored `page_number` (1-based), `source_file`, and `doc_type` on each page before indexing. Those fields now come from the classifier instead of a handwritten list. `is_new_doc` and `page_in_doc` travel with them. This cell stops at the tagged pages. It does not build the vector index.


In [7]:
# ============================================
# STEP 6: Stamp metadata onto each page
# ============================================
#
# WHAT WE'RE DOING:
# Copying the segmentation onto the metadata dict
# the tagging notebook attached before indexing.
#
# WHY THIS MATTERS:
# Filtered retrieval needs doc_type on the page,
# and the page number has to point back at the PDF.
#
# WHAT YOU'LL SEE:
# Metadata and a short text preview for the first two pages.
# ============================================

tagged_pages = []
for row, page in zip(results, doc_pages):
    tagged_pages.append({
        "metadata": {
            "page_number": row["page"] + 1,
            "source_file": PDF_PATH.name,
            "doc_type": row["doc_type"],
            "is_new_doc": row["is_new_doc"],
            "page_in_doc": row["page_in_doc"],
        },
        "text": page["text"],
    })

for item in tagged_pages[:2]:
    print(item["metadata"])
    print(item["text"][:150], "\n---\n")

print(f"Tagged {len(tagged_pages)} pages")


{'page_number': 1, 'source_file': 'pharma-blob-sample.pdf', 'doc_type': 'Cover Letter', 'is_new_doc': 'Yes', 'page_in_doc': 0}
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It Ma 
---

{'page_number': 2, 'source_file': 'pharma-blob-sample.pdf', 'doc_type': 'Certificate of Quality', 'is_new_doc': 'Yes', 'page_in_doc': 0}
Certificate of Quality
This product is manufactured in compliance with our ISO 9001 certified quality management system.
Issued by Cytiva Westborough  
---

Tagged 10 pages
